In my own computations (by hand), I forgot to use the inner product that satisfies the Morimoto criterion, hence the $c_{ijk}$ in addition to the $f_{ijk}$. When restricted to $V=\text{span}_\mathbb{R}(\varepsilon_1,\ldots,\varepsilon_{2m})$, the $\varepsilon_i$ for an orthonormal basis with

$$ |\varepsilon_i|^2 = \frac{(i-1)!}{(2m-i)!}\quad \text{hence}\quad |\varepsilon_i^*\wedge \varepsilon_j^*\otimes \varepsilon_k|^2 = \frac{(2m-i)!}{(i-1)!}\frac{(2m-j)!}{(j-1)!}\frac{(k-1)!}{(2m-k)!}$$

This notebook will compute the conditions under which $f^{ijk}\varepsilon_{i}^*\otimes\varepsilon_{j}^*\otimes\varepsilon_{k}$ is in $\text{Im}(S^1)^\perp$, where $S^1:V^*\otimes\mathfrak{gl}(2,\mathbb{R})\to V^* \wedge V^*\otimes V$ is the first spencer operator, defined by

$$S^1\phi(v_1,v_2)=\phi(v_1)(v_2)-\phi(v_2)(v_1)$$

Then, these conditions will be used to produce a basis for $\text{Im}(S^1)^\perp$.

Remark: If I wanted to make this notebook more time efficient, one way I could do that is by replacing any large coefficient dictionaries and using the attribute .as_coefficient_dict for add objects...A new function substitute_f_for_c would not be so hard to implement

In [1]:
from sympy import *
import time
from collections import defaultdict
import copy
import pickle

In [2]:
# We'll work with gl2+heis(2*m+1)
m=3

In [3]:
# i=Idx('i',range=(1,2*m+1))
# j=Idx('j',range=(1,2*m+1))
# k=Idx('k',range=(1,2*m+1))
c=IndexedBase('c',shape=(2*m,2*m,2*m))
f=IndexedBase('f',shape=(2*m,2*m,2*m))
e=IndexedBase('e',shape=(2*m,2*m,2*m))
Y,H,E,X=symbols('Y,H,E,X')

# Firstly, set the antisymmetric relations
antisymm_subs={}
for a in range(1,2*m+1):
    for d in range(1,2*m+1):
        antisymm_subs[c[a,a,d]]=0
        antisymm_subs[f[a,a,d]]=0
        for b in range(a+1,2*m+1):
            antisymm_subs[c[b,a,d]]=-c[a,b,d]
            antisymm_subs[f[b,a,d]]=-f[a,b,d]


## Helper Methods

In [4]:
def c_unraveled_index(c1):
    '''c1: An index between 0 and 2*m+1
       returns: the unraveled index of c1 in c'''
    ctr=0
    while c1!=c[ctr//((2*m+1)**2),(ctr//(2*m+1))%(2*m+1),ctr%(2*m+1)]:
        ctr+=1
    return ctr

In [5]:
def f_index_tuple(f_elt):
    '''for argument f[i,j,k], returns (i,j,k)'''
    ctr=0
    i=0
    j=0
    k=0
    while f[i,j,k]!=f_elt:
        ctr+=1
        k=(ctr%(2*m))+1
        j=((ctr//(2*m))%(2*m))+1
        i=(ctr//((2*m)**2))+1
    return (i,j,k)

In [6]:
def update_subs_dicts(subs_dict,gl_elt):
    '''subs_dict: A dict representing a cijk substitution
       result: None
       updates the dicts c_substitutions and total_c_subs, as well
       as determined_cijk'''
    for A in [Y,H,E,X]:
        for key in c_substitutions[A]:
            # 'Back substitute'
            if hasattr(c_substitutions[A][key],'subs'):
                c_substitutions[A][key]=c_substitutions[A][key].subs(subs_dict)
            # Add the substitution to the dict gl_elt
        if A==gl_elt: c_substitutions[A].update(subs_dict)
    for key in total_c_subs:
        if hasattr(total_c_subs[key],'subs'):
            total_c_subs[key]=total_c_subs[key].subs(subs_dict)
    total_c_subs.update(subs_dict)
    for key in subs_dict:
        determined_cijk.add(key)
        t=c_index_tuple(key)
        determined_fijk.add(f[t[0],t[1],t[2]])

In [7]:
def c_index_tuple(c_elt):
    '''for argument c[i,j,k], returns (i,j,k)'''
    ctr=0
    i=0
    j=0
    k=0
    while c[i,j,k]!=c_elt:
        ctr+=1
        k=(ctr%(2*m))+1
        j=((ctr//(2*m))%(2*m))+1
        i=(ctr//((2*m)**2))+1
    return (i,j,k)

In [8]:
def sq_len(i,j,k):
    '''returns |eijk|**2'''
    return (factorial(k-1)*factorial(2*m-i)*factorial(2*m-j)/
            (factorial(2*m-k)*factorial(i-1)*factorial(j-1)))

# Computations

In [9]:
# rels[A][i] is the relation obtained from <f,S^1(ei x A)>=0
rels={Y:[0],H:[0],E:[0],X:[0]}

for i in range(1,2*m+1):
    rels[E].append((sum([c[i,j,j] for j in range(1,2*m+1)])).subs(antisymm_subs))
    rels[X].append((sum([c[i,j,j+1] for j in range(1,2*m)])).subs(antisymm_subs))
    rels[H].append((sum([(2*j-2*m-1)*c[i,j,j] for j in range(1,2*m+1)])).subs(antisymm_subs))
    rels[Y].append((sum([(j-1)*(2*m+1-j)*c[i,j,j-1] for j in range(2,2*m+1)])).subs(antisymm_subs))
    

In [10]:
# This dict substitutes in fijk for cijk with the appropriate coefficients
c_to_f_subs={}
for i in range(1,2*m+1):
    for j in range(i+1,2*m+1):
        for k in range(1,2*m+1):
            c_to_f_subs[c[i,j,k]]=f[i,j,k]*sq_len(i,j,k)

In [11]:
# Let's use the relations to construct substitution dictionaries
# which give the determined_cijk in terms of the free_cijk
# The determined cijk will be the keys of total_c_subs

total_c_subs={}
c_substitutions={'antisymm':{},Y:{},H:{},E:{},X:{}}
f_substitutions={'antisymm':{},Y:{},H:{},E:{},X:{}}
determined_cijk=set()
determined_fijk=set()

# Firstly, set the antisymmetric relations as c_substitutions
for a in range(1,2*m+1):
    for d in range(1,2*m+1):
        c_substitutions['antisymm'][c[a,a,d]]=0
        f_substitutions['antisymm'][f[a,a,d]]=0
        for b in range(a+1,2*m+1):
            c_substitutions['antisymm'][c[b,a,d]]=-c[a,b,d]
            f_substitutions['antisymm'][f[b,a,d]]=-f[a,b,d]
total_c_subs.update(c_substitutions['antisymm'])

for A in [E,X,H,Y]:
    for r in range(1,len(rels[A])):
        relevant_cijk=list(rels[A][r].as_coefficients_dict().keys())
        relevant_cijk.sort(key=c_unraveled_index)
        ctr=0
        while True:
            # Find the least cijk in relevant_cijk which is not yet determined
            # Note that rels[A][r] contains only cijk with i<j

            if not (relevant_cijk[ctr] in total_c_subs):
                new_cijk=relevant_cijk[ctr]
                
                # Add the relation to c_substitutions
                new_subs_dict={new_cijk:solve(rels[A][r].subs(total_c_subs),new_cijk)[0]}
                update_subs_dicts(new_subs_dict,A)
                break
            else: 
                ctr+=1

In [12]:
## Generating a basis for the orthocomplement of S1
free_cijk={c[i,j,k] for i in range(1,2*m+1) for j in range(i+1,2*m+1)
           for k in range(1,2*m+1)}.difference(determined_cijk)

free_fijk={f[i,j,k] for i in range(1,2*m+1) for j in range(i+1,2*m+1)
           for k in range(1,2*m+1)}.difference(determined_fijk)

In [13]:
# Write the remainder of dict f_substitutions using c_substitutions

for A in [E,X,H,Y]:
    for key in c_substitutions[A]:
        t=c_index_tuple(key)
        val=c_substitutions[A][key].subs(c_to_f_subs)/sq_len(t[0],t[1],t[2])
        f_substitutions[A][f[t[0],t[1],t[2]]]=val

# Combine all the subs for f other than antisymm subs
all_f_subs={} # Excluding antisymm subs
for A in [E,X,H,Y]:
    all_f_subs.update(f_substitutions[A])

In [14]:
basis_dict={}

for f_fijk in free_fijk:
    t=f_index_tuple(f_fijk)
    basis_dict[f_fijk]=e[t[0],t[1],t[2]]

for d_fijk in determined_fijk:
    t=f_index_tuple(d_fijk)
    for f_fijk in all_f_subs[d_fijk].as_coefficients_dict():
        coeff=all_f_subs[d_fijk].as_coefficients_dict()[f_fijk]
        basis_dict[f_fijk]+=coeff*e[t[0],t[1],t[2]]

In [15]:
S1_perp_basis=list(basis_dict.values())

In [16]:
# pickle S1_perp_basis 

file=open('S1_perp_m%d'%m,'wb') # 'wb' means 'write binary mode'
pickle.dump(S1_perp_basis,file)
file.close()

## Printing for Computation Checks

In [17]:
# # For checking my computations of the determined cijk
# A=Y
# for key in c_substitutions[A]:
#     print('i =',list(c_substitutions[A]).index(key)+1)
#     display(key
#             ,c_substitutions[A][key])
#     print('\n\n')

In [18]:
# # For checking my computations of the determined fijk
# A=Y
# for key in f_substitutions[A]:
#     print('i =',list(f_substitutions[A]).index(key)+1)
#     display(key
#             ,f_substitutions[A][key])
#     print('\n\n')

In [19]:
# # For checking the basis

# for i in [2]:
#     for k in range(4,2*m+1):
#         for j in [k]:
#             if f[i,j,k] in basis_dict:
#                 print((i,j,k),':')
#                 display(basis_dict[f[i,j,k]])
#                 print('----------------------------------------')

# Checking the Manually Computed Basis

Below is the basis I have computed by hand for the general $m$ case

In [20]:
def delta(a,b):
    if a==b: return 1
    else: return 0

In [21]:
# For checking, use a bunch of sublists like in the manual computation
manual_basis_segmented=[]

manual_basis_segmented.append([e[i,j,1] for i in range(3,2*m+1) 
                               for j in range(i+1,2*m+1)])

manual_basis_segmented.append([e[i,j,2]-delta(i,3)*e[2,j,1]
                               for i in range(3,2*m+1) for j in range(i+1,2*m+1)])

manual_basis_segmented.append([e[1,j,3] 
                               for j in range(5,2*m+1)])
manual_basis_segmented.append([e[2,j,3]-Rational(2*(2*m-2),(2*m-1))*e[1,j,2]
                               for j in range(5,2*m+1)])
manual_basis_segmented.append([e[3,j,3]+e[1,j,1]-2*e[2,j,2]+delta(j,4)*e[2,3,1]
                               for j in range(4,2*m+1)])
manual_basis_segmented.append([e[i,j,3]-delta(i,4)*e[2,j,1]
                               for i in range(4,2*m+1) for j in range(i+1,2*m+1)])

manual_basis_segmented.append([e[1,j,k]-delta(k,j+1)*Rational((2*m-j)*j,(2*m-2)*2)*e[1,2,3]
                               for k in range(4,2*m+1) for j in range(2,k)])
manual_basis_segmented.append([e[1,j,j]+(j-3)*e[1,2,2]+(2-j)*e[1,3,3]+Rational((2*m-1)*(j-3),(2*m-3)*3)*e[2,3,4]
                               for j in range(4,2*m+1)])
manual_basis_segmented.append([e[1,j,k]-delta(k,j-1)*e[1,4,3]
                               for k in range(4,2*m) for j in range(k+1,2*m+1)])

manual_basis_segmented.append([e[2,j,k]-delta(k,j+1)*Rational((2*m-j)*j,(2*m-3)*3)*e[2,3,4]
                               for k in range(5,2*m+1) for j in range(3,k)])
# # Something is wrong with the one below :(
manual_basis_segmented.append([e[2,j,j]+Rational(3-j,2)*e[1,2,1]+Rational(1-j,2)*e[2,3,3]
                               +Rational((2*m-2)*(j-1),2*m-1)*e[1,3,2]+Rational(-j*(2*m-3)-2*m-1,2*(2*m-1))*e[1,4,3]
                               for j in range(4,2*m+1)])
manual_basis_segmented.append([e[2,j,k]+delta(k,j-1)*(Rational(2*(2*m-2),(2*m-1))*e[1,4,2]-e[2,4,3])
                               for k in range(4,2*m) for j in range(k+1,2*m+1)])

manual_basis_segmented.append([e[3,4,4]-2*e[1,3,1]+3*e[2,3,2]+Rational(6*(2*m-2)-3*(2*m-3),2*m-1)*e[1,4,2]-3*e[2,4,3]])
manual_basis_segmented.append([e[3,j,4]-Rational(3*(2*m-3),2*m-1)*e[1,j,2]+delta(j,5)*e[2,3,1]
                              for j in range(5,2*m+1)])

manual_basis_segmented.append([e[3,j,k]+delta(k,j+1)*(Rational(j*(2*m-j),2*m-1)*(e[1,3,2]-e[1,4,3]))
                              for k in range(5,2*m+1) for j in range(4,k)])
manual_basis_segmented.append([e[3,j,j]+(2-j)*e[1,3,1]+(j-1)*e[2,3,2]+Rational(2*(2*m-2)*(j-1),2*m-1)*e[1,4,2]+(1-j)*e[2,4,3]
                              for j in range(5,2*m+1)])
manual_basis_segmented.append([e[3,j,k]+delta(k,j-1)*e[2,3,1]
                              for k in range(5,2*m) for j in range(k+1,2*m+1)])

manual_basis_segmented.append([e[4,j,4]+2*e[1,j,1]-3*e[2,j,2]+delta(j,5)*e[2,4,1]
                              for j in range(5,2*m+1)])

manual_basis_segmented.append([e[4,j,5]-Rational(4*(2*m-4),2*m-1)*e[1,j,2]+delta(j,5)*(4*e[2,4,2]-3*e[1,4,1])+delta(j,6)*e[2,4,1]
                              for j in range(5,2*m+1)])

manual_basis_segmented.append([e[4,j,k]+delta(k,j+1)*Rational(j*(2*m-j),2*m-1)*e[1,4,2]
                              for k in range(6,2*m+1) for j in range(5,k)])
manual_basis_segmented.append([e[4,j,j]+(2-j)*e[1,4,1]+(j-1)*e[2,4,2]
                              for j in range(6,2*m+1)])
manual_basis_segmented.append([e[4,j,k]+delta(k,j-1)*e[2,4,1]
                               for k in range(6,2*m) for j in range(k+1,2*m+1)])

manual_basis_segmented.append([e[i,j,k]-delta(k,i-1)*e[2,j,1]
                              for i in range(5,2*m+1) for k in range(1,i) for j in range(i+1,2*m+1)])

manual_basis_segmented.append([e[i,j,i]+(i-2)*e[1,j,1]+(1-i)*e[2,j,2]+delta(i,j-1)*e[2,i,1]
                              for i in range(5,2*m+1) for j in range(i+1,2*m+1)])

manual_basis_segmented.append([e[i,j,i+1]-Rational(i*(2*m-i),2*m-1)*e[1,j,2]+delta(j,i+2)*e[2,i,1]+delta(j,i+1)*((1-i)*e[1,i,1]+i*e[2,i,2])
                              for i in range(5,2*m) for j in range(i+1,2*m+1)])

manual_basis_segmented.append([e[i,j,k]+delta(k,j+1)*Rational(j*(2*m-j),2*m-1)*e[1,i,2]
                              for i in range(5,2*m-1) for k in range(i+2,2*m+1) for j in range(i+1,k)])
manual_basis_segmented.append([e[i,j,j]+(2-j)*e[1,i,1]+(j-1)*e[2,i,2]
                              for i in range(5,2*m-1) for j in range(i+2,2*m+1)])
manual_basis_segmented.append([e[i,j,k]+delta(k,j-1)*e[2,i,1]
                              for i in range(5,2*m-1) for k in range(i+2,2*m+1) for j in range(k+1,2*m+1)])

manual_basis=[]
for A in manual_basis_segmented:
    manual_basis+=A

## Basic Testing

In [22]:
# Test: Make sure determined_cijk agree with my computations
test_determined_cijk=set()
for i in range(1,3):
    for j in range(i+1,2*m+1):
        test_determined_cijk.add(c[i,j,1])

for i in range(1,3):
    for j in range(i+1,2*m+1):
        test_determined_cijk.add(c[i,j,2])
        
for t in [(1,2,3),(1,3,3),(1,4,3),(2,3,3),(2,4,3)]:
    test_determined_cijk.add(c[t[0],t[1],t[2]])
    
test_determined_cijk.add(c[(2,3,4)])
    
if test_determined_cijk==determined_cijk:
    print('determined_cijk agrees with my computations')

determined_cijk agrees with my computations


In [23]:
# Testing the manually generated basis
print(set(manual_basis)==set(S1_perp_basis))


# test1=set(manual_basis).difference(set(S1_perp_basis))
# test2=set(S1_perp_basis).difference(set(manual_basis))
# if test1!=set():
#     print(test1,'in manual_basis, but not S1_perp_basis')
# if test2!=set():
#     print(test2,'in S1_perp_basis, but not manual_basis')

True
